In [5]:
import os
import torch
import numpy as np
import pandas as pd
import glob
from sklearn.neighbors import KDTree


In [6]:
from src.data.data import SpacioTemporalDataset

In [19]:
file_list = [
    "P1-Rec1-All-Data-New_Section_1.csv"
]
std = SpacioTemporalDataset(
    root_dir="./data/processed/",
    recursive=True,
)

Window slice(np.int64(59351), np.int64(59352), None) too small for kt=5 and ks=10. Skipping... 
Loaded 2250 graphs from ./data/processed/


In [28]:
import sys

memory_usage = sys.getsizeof(std.graphs)
print(f"Memory usage of 'std': {memory_usage} bytes ({memory_usage / 1024 / 1024:.2f} MB)")

Memory usage of 'std': 18232 bytes (0.02 MB)


In [22]:
std.graphs

[HeteroData(
   node={
     x=[1772, 5],
     num_nodes=1772,
   },
   (node, temporal, node)={ edge_index=[2, 17690] },
   (node, spatial, node)={ edge_index=[2, 35440] }
 ),
 HeteroData(
   node={
     x=[2477, 5],
     num_nodes=2477,
   },
   (node, temporal, node)={ edge_index=[2, 24740] },
   (node, spatial, node)={ edge_index=[2, 49540] }
 ),
 HeteroData(
   node={
     x=[3559, 5],
     num_nodes=3559,
   },
   (node, temporal, node)={ edge_index=[2, 35560] },
   (node, spatial, node)={ edge_index=[2, 71180] }
 ),
 HeteroData(
   node={
     x=[3607, 5],
     num_nodes=3607,
   },
   (node, temporal, node)={ edge_index=[2, 36040] },
   (node, spatial, node)={ edge_index=[2, 72140] }
 ),
 HeteroData(
   node={
     x=[3266, 5],
     num_nodes=3266,
   },
   (node, temporal, node)={ edge_index=[2, 32630] },
   (node, spatial, node)={ edge_index=[2, 65320] }
 ),
 HeteroData(
   node={
     x=[2785, 5],
     num_nodes=2785,
   },
   (node, temporal, node)={ edge_index=[2, 27820] },

In [29]:
g = std.graphs[0]

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
%matplotlib qt
# Extract first 100 nodes
num_nodes = min(100, g["node"].num_nodes)
X_subset = g["node"].x[:num_nodes]

# Filter edges to only include those within first 100 nodes
temporal_edges = g["node", "temporal", "node"].edge_index
spatial_edges = g["node", "spatial", "node"].edge_index

# Filter temporal edges
temporal_mask = (temporal_edges[0] < num_nodes) & (temporal_edges[1] < num_nodes)
temporal_edges_filtered = temporal_edges[:, temporal_mask]

# Filter spatial edges
spatial_mask = (spatial_edges[0] < num_nodes) & (spatial_edges[1] < num_nodes)
spatial_edges_filtered = spatial_edges[:, spatial_mask]

# Create networkx graph
G = nx.Graph()
G.add_nodes_from(range(num_nodes))

# Add edges with types
for i in range(temporal_edges_filtered.shape[1]):
    src, dst = temporal_edges_filtered[0, i].item(), temporal_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='temporal')

for i in range(spatial_edges_filtered.shape[1]):
    src, dst = spatial_edges_filtered[0, i].item(), spatial_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='spatial')

# Create figure
fig, ax = plt.subplots(figsize=(16, 16))

# Use x,y coordinates from node features as positions
pos = {}
for i in range(num_nodes):
    x = X_subset[i, 1].item()  # x coordinate at index 1
    y = X_subset[i, 2].item()  # y coordinate at index 2
    pos[i] = (x, y)

# Draw temporal edges (blue) with arrows from smaller to bigger index
temporal_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'temporal']
for u, v in temporal_edge_list:
    # Ensure arrow goes from smaller to bigger index
    start, end = (u, v) if u < v else (v, u)
    alpha = 0.2 + (start / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    
    # Create arrow patch
    arrow = FancyArrowPatch(pos[start], pos[end],
                           arrowstyle='->', mutation_scale=10, 
                           color='blue', alpha=alpha, linewidth=0.5, zorder=1)
    ax.add_patch(arrow)

# Draw spatial edges (red)
spatial_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'spatial']
for u, v in spatial_edge_list:
    alpha = 0.2 + (u / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
            'r-', alpha=alpha, linewidth=0.5, zorder=1)

# Draw nodes (green) with varying alpha
for node in range(num_nodes):
    alpha = 0.2 + (node / num_nodes) * 0.7
    ax.scatter(pos[node][0], pos[node][1], c='green', s=50, alpha=alpha, zorder=2)

ax.set_title(f'Graph Visualization (First {num_nodes} Nodes)', fontsize=16)
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
plt.legend()
plt.tight_layout()

# Display in separate window
plt.show()

print(f"Visualized {num_nodes} nodes")
print(f"Temporal edges: {len(temporal_edge_list)}")
print(f"Spatial edges: {len(spatial_edge_list)}")

Visualized 100 nodes
Temporal edges: 225
Spatial edges: 423


/tmp/ipykernel_3109774/1599774037.py:74: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()
